# Lecture 15: Intro to Pytorch (GPU)

PyTorch is a popular deep learning framework.
It is very similar to Needle!

**Important**: before running this notebook, please change your Runtime type to a type that supports GPU acceleration (ideally "cuda").
As of 2026, a good lightweight choice (with good availability) is "T4 GPU".

To do this, in the upper-right corner of Colab, there is a dropdown arrow (to the right of the "RAM, Disk" resource monitor icon).
Click that, and click **"Change runtime type"**.

<img src=figures/colab_change_runtime_type.png alt="How to change Runtime type in Colab (step 1)" width="40%">

Then, choose "T4 GPU":

<img src=figures/colab_change_runtime_type_2.png alt="How to change Runtime type in Colab (step 2)" width="40%">

In [16]:
# first some imports/helper functions
import math
import time

from typing import Any

import torch
import torchvision
from torch.utils.data import DataLoader

def get_model_device(model: torch.nn.Module) -> torch.device:
    # Heuristic: use the device of the first model parameter
    # Note: this won't work correctly for models that are sharded at
    # the op level (ie FSDP)!
    return next(model.parameters()).device

def count_model_parameters(model: torch.nn.Module) -> int:
    """Counts the number of trainable parameters in a model.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# GPU Acceleration

Due to the computationally demanding nature of deep learning, it's common to employ GPU acceleration to speed up model training and inference.

In practice, it's possible to see up to **10x - 100x** speedups when using the GPU vs  the CPU!

While the needle framework (as covered in HW0 - HW2) operated purely on the CPU, pytorch supports the GPU for many of its core operations.

## Tensor devices

An important concept is `Tensor.device`.
All Tensors (whether they are input Tensors, intermediate activation Tensors, parameter Tensors, or gradient Tensors) live on a device.

By default, `torch.Tensor`'s live on the CPU device.

However, if your machine has access to a GPU, it's possible to move that `torch.Tensor` to the GPU device, via the `torch.Tensor::to(device=...)` method:

Tip: device `"cuda:0"` means: an Nvidia GPU card ("cuda") at device index `0`.
A machine can have multiple GPU's connected to it.
If a machine had 4 cuda GPU's attached to it, then to pytorch, there would be four cuda devices: `"cuda:0", "cuda:1", "cuda:2", "cuda:3"`.
This is known as "multi-GPU" training, which is an important part of scaling up DNN model training.

But, in this notebook (and most of this class), we'll focus on just single-GPU training.

In [2]:
# Ensure that your Colab instance type is "T4 GPU"!
if not torch.accelerator.is_available():
    raise RuntimeError(f"Please switch to a Colab runtime type that supports acceleration. Ex: T4 GPU")
gpu_device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {gpu_device} device")

# by default, x lives on the cpu
x = torch.tensor([1, 2, 3])
x = x.to(device=gpu_device)

# x now lives on the gpu!
print(f"x.device: {x.device}")

Using cuda device
x.device: cuda:0


When performing operations between two `torch.Tensor`'s (ex: matrix multiply, or element-wise sum), pytorch requires that both `torch.Tensor`'s must reside on the same device.

If they don't, you'll see an error thrown:

In [3]:
x2 = torch.tensor([4, 5, 6], dtype=torch.float32)
# throws an error, since `x` is on the gpu
print(x + x2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [4]:
# to fix this, we must also move `x2` to the gpu
x2 = x2.to(device=gpu_device)
# works now!
print(x + x2)

tensor([5., 7., 9.], device='cuda:0')


## Module devices

PyTorch makes it easy to move a Module (eg `torch.nn.Linear`) to different devices.

For instance, recall that `torch.nn.Linear` is a module that has a `weight` and `bias` Parameter:

In [5]:
linear = torch.nn.Linear(in_features=3, out_features=8)
print(linear.weight)
print(linear.bias)

Parameter containing:
tensor([[ 0.3391,  0.4690,  0.0701],
        [ 0.0090,  0.2321, -0.1987],
        [-0.5200, -0.2145, -0.2675],
        [ 0.1766,  0.1442, -0.3399],
        [-0.0021, -0.2714,  0.0612],
        [-0.4531,  0.0299, -0.4963],
        [-0.4651,  0.3592,  0.2963],
        [ 0.5764,  0.3063, -0.1381]], requires_grad=True)
Parameter containing:
tensor([ 0.1265, -0.2338, -0.3121, -0.2795, -0.0927, -0.3080, -0.0341,  0.4739],
       requires_grad=True)


By default, the Linear layer resides on the CPU device (because its weight/bias Parameters live on the CPU device).

To move the Linear layer to the GPU device, we can use the `torch.nn.Module::to(device=...)` method:

In [6]:
linear = linear.to(device=gpu_device)
print(linear)
print(linear.weight)
print(linear.bias)

Linear(in_features=3, out_features=8, bias=True)
Parameter containing:
tensor([[ 0.3391,  0.4690,  0.0701],
        [ 0.0090,  0.2321, -0.1987],
        [-0.5200, -0.2145, -0.2675],
        [ 0.1766,  0.1442, -0.3399],
        [-0.0021, -0.2714,  0.0612],
        [-0.4531,  0.0299, -0.4963],
        [-0.4651,  0.3592,  0.2963],
        [ 0.5764,  0.3063, -0.1381]], device='cuda:0', requires_grad=True)
Parameter containing:
tensor([ 0.1265, -0.2338, -0.3121, -0.2795, -0.0927, -0.3080, -0.0341,  0.4739],
       device='cuda:0', requires_grad=True)


Now that our Linear layer is on the GPU device, we can run its forward() method with a GPU Tensor:

In [7]:
x = torch.rand(size=(2, 3), dtype=torch.float32).to(device=gpu_device)
out = linear(x)
print(out)

tensor([[ 4.8567e-01, -3.7384e-01, -9.2828e-01, -4.5700e-01, -8.1845e-02,
         -1.0531e+00,  8.6859e-05,  7.6623e-01],
        [ 5.8389e-01, -4.6864e-02, -5.9856e-01, -1.4165e-01, -3.1734e-01,
         -3.8809e-01,  1.9410e-01,  8.2952e-01]], device='cuda:0',
       grad_fn=<AddmmBackward0>)


In [8]:
# Note that trying to pass a CPU tensor to the GPU Linear layer will result in an error!
x_cpu = torch.rand(size=(2, 3), dtype=torch.float32)
out = linear(x_cpu)
print(out)

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_addmm)

## Demo: MLP CIFAR10 with GPU

To demonstrate the benefits that GPU-accelerated training can give us, let's compare training our MLP classifier on CIFAR-10 with and without GPU acceleration.

Most of the below code is copy+pasted from the other notebook.

In [10]:
class LinearBlock(torch.nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.linear = torch.nn.Linear(in_features=in_features, out_features=out_features)
        self.layer_norm = torch.nn.LayerNorm(normalized_shape=out_features)
        self.relu = torch.nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.linear(x)
        out = self.layer_norm(out)
        out = self.relu(out)
        return out

class MyMLPV2(torch.nn.Module):
    # V2: use the LinearBlock
    def __init__(self, num_input_feats: int, num_classes: int, num_blocks: int = 2, hidden_dim: int = 8):
        super().__init__()
        self.num_classes = num_classes
        self.num_blocks = num_blocks
        self.hidden_dim = hidden_dim
        # first linear is "special": num_input_feats -> hidden_dim
        linear1 = torch.nn.Linear(in_features=num_input_feats, out_features=hidden_dim)
        # remaining linear blocks operate on hidden_dim -> hidden_dim
        # Create LinearBlocks
        if False:
            # (impl1)
            modules = [linear1]
            for _ in range(self.num_blocks):
                modules.append(LinearBlock(in_features=hidden_dim, out_features=hidden_dim))
            self.linear_blocks = torch.nn.Sequential(*modules)
        else:
            # (impl2) a slightly more compact implementation, using list comprehensions
            self.linear_blocks = torch.nn.Sequential(
                linear1,
                *[
                    LinearBlock(in_features=hidden_dim, out_features=hidden_dim)
                    for _ in range(self.num_blocks)
                ]
            )
        # final linear block produces logits
        self.linear_cls = torch.nn.Linear(in_features=hidden_dim, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.linear_blocks(x)
        logits = self.linear_cls(out)
        return logits

In [ ]:
# Train loop
def train_epoch(dataloader, model: torch.nn.Module, loss_fn, optimizer, log_every_n_steps: int = 100):
    size = len(dataloader.dataset)
    model.train()
    model_device = get_model_device(model)
    tic_total = time.time()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction error
        # [B, C, H, W] -> [B, C*H*W]
        X = X.flatten(start_dim=1)
        # Important: move inputs/labels to same device as model device
        X, y = X.to(model_device), y.to(model_device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % log_every_n_steps == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    dur_total = time.time() - tic_total
    tput_total = size / dur_total
    print(f"Total training time: {dur_total} secs (throughput={tput_total} samples/sec)")

# Evaluation
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    model_device = get_model_device(model)
    test_loss, correct = 0, 0
    tic_total = time.time()
    with torch.no_grad():
        for X, y in dataloader:
            # [B, C, H, W] -> [B, C*H*W]
            X = X.flatten(start_dim=1)
            # Important: move inputs/labels to same device as model device
            X, y = X.to(model_device), y.to(model_device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    dur_total = time.time() - tic_total
    tput_total = size / dur_total
    print(f"Total test time: {dur_total} secs (throughput={tput_total} samples/sec)")

# Download training data
training_data_cifar10 = torchvision.datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=torchvision.transforms.ToTensor(),
)

# Download test data from open datasets.
test_data_cifar10 = torchvision.datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=torchvision.transforms.ToTensor(),
)


In [18]:
batch_size_cifar10 = 512

# Create data loaders.
train_dataloader_cifar10 = DataLoader(training_data_cifar10, batch_size=batch_size_cifar10)
test_dataloader_cifar10 = DataLoader(test_data_cifar10, batch_size=batch_size_cifar10)

## CPU

In [32]:
# Create model
model = MyMLPV2(num_input_feats=3*32*32, num_classes=10, hidden_dim=1024, num_blocks=4)
print(model)
print(f"Num model parameters: {count_model_parameters(model)}")

optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

# Run training
epochs = 1
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_epoch(train_dataloader_cifar10, model, loss_fn, optimizer)
    test(test_dataloader_cifar10, model, loss_fn)
print("Done!")

MyMLPV2(
  (linear_blocks): Sequential(
    (0): Linear(in_features=3072, out_features=1024, bias=True)
    (1): LinearBlock(
      (linear): Linear(in_features=1024, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (2): LinearBlock(
      (linear): Linear(in_features=1024, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (3): LinearBlock(
      (linear): Linear(in_features=1024, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (4): LinearBlock(
      (linear): Linear(in_features=1024, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
  )
  (linear_cls): Linear(in_features=1024, out_features=10, bias=True)
)
Num model parameters: 7363594
Epoch 1
---

## GPU

In [33]:
# Create model
model = MyMLPV2(num_input_feats=3*32*32, num_classes=10, hidden_dim=1024, num_blocks=4).to(device=gpu_device)
print(model)
print(f"Num model parameters: {count_model_parameters(model)}")

optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

# Run training
epochs = 1
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_epoch(train_dataloader_cifar10, model, loss_fn, optimizer)
    test(test_dataloader_cifar10, model, loss_fn)
print("Done!")

MyMLPV2(
  (linear_blocks): Sequential(
    (0): Linear(in_features=3072, out_features=1024, bias=True)
    (1): LinearBlock(
      (linear): Linear(in_features=1024, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (2): LinearBlock(
      (linear): Linear(in_features=1024, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (3): LinearBlock(
      (linear): Linear(in_features=1024, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (4): LinearBlock(
      (linear): Linear(in_features=1024, out_features=1024, bias=True)
      (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
  )
  (linear_cls): Linear(in_features=1024, out_features=10, bias=True)
)
Num model parameters: 7363594
Epoch 1
---

To really showcase the power of GPU acceleration, here I have increased the model parameters by ~10x (7363594 -> 73521162) by making the following changes:
- hidden_dim 1024 -> 2048
- 4x'd the number of LinearBlocks

Despite the dramatically increased computation, the training time is only ~20% longer (8.6 secs vs 7.2 secs)!

In [35]:
# Create model
model = MyMLPV2(num_input_feats=3*32*32, num_classes=10, hidden_dim=2048, num_blocks=16).to(device=gpu_device)
print(model)
print(f"Num model parameters: {count_model_parameters(model)}")

optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

# Run training
epochs = 1
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_epoch(train_dataloader_cifar10, model, loss_fn, optimizer)
    test(test_dataloader_cifar10, model, loss_fn)
print("Done!")

MyMLPV2(
  (linear_blocks): Sequential(
    (0): Linear(in_features=3072, out_features=2048, bias=True)
    (1): LinearBlock(
      (linear): Linear(in_features=2048, out_features=2048, bias=True)
      (layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (2): LinearBlock(
      (linear): Linear(in_features=2048, out_features=2048, bias=True)
      (layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (3): LinearBlock(
      (linear): Linear(in_features=2048, out_features=2048, bias=True)
      (layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (4): LinearBlock(
      (linear): Linear(in_features=2048, out_features=2048, bias=True)
      (layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (relu): ReLU()
    )
    (5): LinearBlock(
      (linear): Linear(in_features=2048, out_features=2048, bias=True)
      (layer_norm): Lay

# Takeaways

meow